# Practical 5 – Content-Based Recommender (Amazon Reviews 2023, All_Beauty)
Dataset: https://amazon-reviews-2023.github.io/ – files `All_Beauty.jsonl.gz` and `meta_All_Beauty.jsonl.gz`

**Aim:** TF-IDF product vectors → cosine similarity (manual) → rating-weighted user profile → Top-N → explanation → Hit@K / MRR / NDCG.

## 1. Setup

In [3]:
from google.colab import drive; drive.mount("/content/drive")
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

D = "/content/drive/MyDrive/amazon_reviews"
TOP_N = K = 10                      # list size / ranking cut-off
MAX_PRODUCTS, MIN_INTER, MAX_EVAL = 5000, 5, 300

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load and filter (most-reviewed products, users with ≥5 interactions)

In [4]:
rev = pd.read_json(f"{D}/All_Beauty.jsonl.gz", lines=True, compression="gzip")
rev = rev[["user_id", "parent_asin", "rating", "timestamp"]].dropna()
meta = pd.read_json(f"{D}/meta_All_Beauty.jsonl.gz", lines=True, compression="gzip")
meta = (meta[["parent_asin", "title", "main_category", "store", "categories", "features", "description"]]
        .dropna(subset=["parent_asin", "title"]).drop_duplicates("parent_asin"))

rev = rev.sort_values("timestamp").drop_duplicates(["user_id", "parent_asin"], keep="last")   # latest repeat only
rev = rev[rev.parent_asin.isin(rev.parent_asin.value_counts().head(MAX_PRODUCTS).index)]
rev = rev[rev.groupby("user_id").user_id.transform("size") >= MIN_INTER]
meta = meta[meta.parent_asin.isin(rev.parent_asin)].reset_index(drop=True)
rev = rev[rev.parent_asin.isin(meta.parent_asin)]
print(rev.user_id.nunique(), "users |", len(meta), "products |", len(rev), "ratings")

159 users | 495 products | 1167 ratings


## 3. Product content → TF-IDF

In [5]:
to_text = lambda v: " ".join(map(str, v)) if isinstance(v, list) else ("" if pd.isna(v) else str(v))
cols = ["title", "main_category", "store", "categories", "features", "description"]
meta["content"] = (meta[cols].apply(lambda c: c.map(to_text)).apply(" ".join, axis=1)
                   .str.lower().str.replace(r"\s+", " ", regex=True))

tfidf = TfidfVectorizer(stop_words="english", min_df=2, max_features=30000)
X = tfidf.fit_transform(meta.content)            # one row per product (rows are L2-normalised)
terms = tfidf.get_feature_names_out()
pid = {p: i for i, p in enumerate(meta.parent_asin)}
print("TF-IDF matrix:", X.shape)

TF-IDF matrix: (495, 2886)


## 4. Manual cosine similarity + Top-N helper

In [6]:
def cosine(q, M):
    """Cosine similarity between one sparse row q and every row of M."""
    dots  = (M @ q.T).toarray().ravel()
    norms = np.sqrt(M.multiply(M).sum(1)).A1 * np.sqrt(q.multiply(q).sum())
    return np.divide(dots, norms, out=np.zeros_like(dots), where=norms > 0)

def top(scores, n, exclude=()):
    s = scores.copy(); s[list(exclude)] = -np.inf
    i = np.argsort(-s)[:n]
    return meta.iloc[i][["parent_asin", "title", "store"]].assign(score=s[i]).reset_index(drop=True)

def similar(asin, n=TOP_N):                      # item-to-item
    i = pid[asin]
    return top(cosine(X[i], X), n, [i])

## 5. Product-to-product similarity

In [7]:
print("Target product:", meta.title[0])
display(similar(meta.parent_asin[0]))

Target product: Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin, 100% Pure and Natural, Hexane-Free, Moisturizing, For Healthy Skin, Silky Hair, Multiuse Body Oil, 16 fl. Oz X 2


,parent_asin,title,store,score
0,B015WVW8EW,"PURE Castor Oil with Sweet Almond Oil, Coconut...",Mia Adora,0.403245
1,B01I2MU2JQ,Sky Organics Fractionated Coconut Oil for Body...,Sky Organics,0.399037
2,B0190JGPZA,Organic Tea Tree Oil Conditioner : #1 with Tea...,Keeva Organics,0.304855
3,B0190K63GA,"Best Anti-bacterial Body Wash : Tea Tree Oil, ...",Keeva Organics,0.303089
4,B00ZPQB9RW,"Gio Naturals Organic Moroccan Argan Oil, 4 oz ...",None,0.291963
5,B01LFO6IZK,Inn&Co Highest Quality Organic Moringa Oil - 4...,Inn&Co,0.265308
6,B00U2VQZC4,Neutrogena Ultra Light Facial Cleansing Oil & ...,Neutrogena,0.258370
7,B086VYKNDF,Aloe Vera Gel 100% Natural Moisturisers - Pure...,CIMIY,0.254800
8,B081632HX6,Livordo Moroccan Argan Oil Essential Organic C...,Livordo,0.249210
9,B00OXDWFG2,"Nadira Organics Virgin Argan Oil for Skin, Fac...",UNGLINGA,0.247953


## 6. Chronological split + rating-weighted user profile
Latest interaction per user → test. Weight of each training item = `max(rating − user_mean, 0)`; profile = weighted average of the item TF-IDF vectors.

In [8]:
test  = rev.loc[rev.groupby("user_id").timestamp.idxmax()]
train = rev.drop(test.index)
by_user = {u: g for u, g in train.groupby("user_id")}
print("train:", len(train), "| test:", len(test))

def profile(u):
    h = by_user[u]; r = h.rating.to_numpy(float)
    w = np.maximum(r - r.mean(), 0)
    if w.sum() == 0: w = r / 5                   # user rated everything the same
    rows = h.parent_asin.map(pid).to_numpy()
    return csr_matrix(X[rows].T @ w / w.sum()), rows

def recommend(u, n=TOP_N):
    p, seen = profile(u)
    return top(cosine(p, X), n, seen)            # seen items excluded

def top_terms(p, n=15):
    v = p.toarray().ravel(); i = np.argsort(-v)[:n]
    return pd.DataFrame({"term": terms[i], "weight": v[i]})

def explain(p, asin, n=8):                       # per-term contribution to the match
    c = X[pid[asin]].multiply(p).toarray().ravel()
    i = np.argsort(-c)[:n]; i = i[c[i] > 0]
    return pd.DataFrame({"term": terms[i], "contribution": c[i]})

train: 1008 | test: 159


## 7. Demo user: profile → recommendations → explanation

In [9]:
u = train.user_id.iloc[0]
p, _ = profile(u)
print("Strongest profile terms for the user"); display(top_terms(p))

recs = recommend(u)
print(f"Top {TOP_N} recommendations"); display(recs)

print("Why #1:", recs.title[0]); display(explain(p, recs.parent_asin[0]))

Strongest profile terms for the user


,term,weight
0,hair,0.256466
1,clips,0.184391
2,pcs,0.118174
3,barrettes,0.083036
4,scrunchies,0.081258
5,snap,0.079125
6,rhinestone,0.079097
7,nail,0.068946
8,file,0.068918
9,women,0.061270


Top 10 recommendations


,parent_asin,title,store,score
0,B07VWLSNYN,"UMICKOO Hair Straightener,Upgraded Flat Iron f...",UMIKOO,0.376721
1,B07DQGT2JH,4 Pieces Large Grip Octopus Clip No Slip Spide...,Bememo,0.359977
2,B0B4KS62Z8,L'ANGE HAIR Rubber Sectioning Hair Clips | Nar...,L'ANGE HAIR,0.320102
3,B07DPKZPRH,PARWIN Infrared Flat Iron 1 1/4 Inch Digital H...,PARWIN,0.317584
4,B07QYD1J5C,"AIRUISI Ceramic Flat Iron for Hair, Profession...",AIRUISI,0.304889
5,B07HR754WV,NeWisdom 72 PCS Toddler Girls Hair Accessory S...,NeWisdom,0.298016
6,B0919LJ9WT,"18 Pack Hair Ties,Spiral elastics, Multi Color...",Gomayee,0.288405
7,B075D96DZK,30 PCS Blue Hair Care Rollers Hair Curlers Sil...,ONEONEY,0.283263
8,B079R2VWNX,Spring Twist hair 3PCS Crochet Braids Hair Syn...,Becolor,0.270376
9,B00AX3YNPC,Braun Silk Epil Female Epilator Se7921spa 1 Count,Braun,0.266181


Why #1: UMICKOO Hair Straightener,Upgraded Flat Iron for Hair With Real-Time Temperature Display And Adjustable Temperature,1 Inch (Rose Gold)


,term,contribution
0,hair,0.112448
1,temperature,0.013856
2,clips,0.013458
3,straightener,0.008161
4,adjustable,0.004511
5,floating,0.003989
6,display,0.003835
7,different,0.002997


## 8. Ranking evaluation (held-out items rated ≥ 4 are relevant)

In [10]:
ev = test[test.rating >= 4]
ev = ev.sample(min(MAX_EVAL, len(ev)), random_state=42)

ranks = []
for u, a in zip(ev.user_id, ev.parent_asin):
    p, seen = profile(u)
    s = cosine(p, X); s[seen] = -np.inf
    ranks.append((s > s[pid[a]]).sum() + 1)      # rank of the held-out item among unseen products
ranks = np.array(ranks); hit = ranks <= K

print(f"Users: {len(ranks)}")
print(f"Hit@{K}:  {hit.mean():.4f}")
print(f"MRR@{K}:  {np.where(hit, 1 / ranks, 0).mean():.4f}")
print(f"NDCG@{K}: {np.where(hit, 1 / np.log2(ranks + 1), 0).mean():.4f}")

Users: 133
Hit@10:  0.0602
MRR@10:  0.0170
NDCG@10: 0.0272
